Imports:

In [1]:
import gzip
import os
import nltk
import re
from gensim.models import Word2Vec

### We used the dutch OpenSubtitles Corpus as it matches our goal for TV emotion classification


- Source: [OpenSubtitles Dutch corpus (OPUS collection)](https://opus.nlpl.eu/OpenSubtitles/nl&en/v2024/OpenSubtitles)



In [ ]:

gz_file_path = "nl.txt.gz"

def extract_and_examine_corpus(gz_file_path):
    """Extract and examine the corpus"""
    
    output_file = gz_file_path.replace('.gz', '')
    
    with gzip.open(gz_file_path, 'rt', encoding='utf-8') as f_in:
        with open(output_file, 'w', encoding='utf-8') as f_out:
            f_out.write(f_in.read())
    
    # Count total lines
    with open(output_file, 'r', encoding='utf-8') as f:
        line_count = sum(1 for _ in f)
    
    print(f"\nTotal lines in corpus: {line_count:,}")
    return output_file

In [4]:
corpus_file = extract_and_examine_corpus("nl.txt.gz")

Sample lines from corpus:
1: WERKKAMP CHANGZHOU JIANGSU, CHINA...
2: VACCINATIEPROGRAMMA WERELDGEZONDHEIDSORGANISATIE...
3: Wat is er?...
4: Wat is er mis?...
5: - Ga weg bij haar....
6: Blijf uit haar buurt....
7: Bel een ambulance....
8: Ze heeft een hartstilstand....
9: - Ze kan niet weg....
10: We gaan naar het ziekenhuis....

Total lines in corpus: 203,754,809


In [ ]:
# Download Dutch tokenizer (run once)
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/florislokhorst/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

### Some quick cleaning to reduce noise and increase relevance:

In [ ]:
def minimal_clean(text):
    """Quick cleaning - just the essentials"""
    import re
    
    text = re.sub(r'<[^>]+>', '', text)  # HTML tags
    text = re.sub(r'\[.*?\]', '', text)  # [MUSIC], etc
    text = re.sub(r'\d+:\d+:\d+', '', text)  # timestamps
    text = re.sub(r'\s+', ' ', text)  # normalize spaces
    
    return text.strip()



In [9]:
sentences = []
with open("nl.txt", 'r', encoding='utf-8') as f:
    for line in f:
        cleaned = minimal_clean(line)
        if len(cleaned) > 10:
            words = cleaned.split()
            if len(words) > 3:
                sentences.append(words)
                

print(f"Got {len(sentences)} sentences")

Got 141459570 sentences


### Training and parameter tuning:

In [22]:
#limiting sentences for training speed

training_sentences = sentences[:3000000]


In [21]:
def train_and_evaluate_model(sentences, params, name):
    """Train model with specific parameters and evaluate"""
    print(f"\n=== Training {name} ===")
    print(f"Parameters: {params}")
    
    model = Word2Vec(sentences=sentences, **params)
    
    # Quick evaluation
    test_words = ['goed', 'slecht', 'mooi', 'blij']
    print(f"Vocabulary size: {len(model.wv.index_to_key)}")
    
    for word in test_words:
        if word in model.wv:
            similar = model.wv.most_similar(word, topn=3)
            print(f"'{word}': {[w for w, score in similar]}")
    
    return model

# Test different configurations
configs = {
    "Baseline": {"vector_size": 200, "window": 5, "min_count": 5, "sg": 1, "epochs": 10, "workers": 4},
    "larger_vector_size": {"vector_size": 500, "window": 5, "min_count": 5, "sg": 1, "epochs": 10, "workers": 4},
    "smaller_window": {"vector_size": 200, "window": 3, "min_count": 5, "sg": 1, "epochs": 10, "workers": 4},
    "higher_mincount": {"vector_size": 200, "window": 5, "min_count": 10, "sg": 1, "epochs": 10, "workers": 4},
    "cbow_model": {"vector_size": 200, "window": 5, "min_count": 5, "sg": 0, "epochs": 10, "workers": 4},
    "Higher_epochs": {"vector_size": 200, "window": 5, "min_count": 5, "sg": 0, "epochs": 20, "workers": 4}
}

models = {}
for name, params in configs.items():
    models[name] = train_and_evaluate_model(training_sentences, params, name)


=== Training Baseline ===
Parameters: {'vector_size': 200, 'window': 5, 'min_count': 5, 'sg': 1, 'epochs': 10, 'workers': 4}


KeyboardInterrupt: 

Parameter Justification:
- vector_size = 200: Balance between expressiveness and efficiency
- window = 5: Captures sufficient conversational context
- min_count = 10: Filters noise, reduces vocab from 52K to 30K words
- sg = 1 (Skip-gram): Better for rare emotional vocabulary than CBOW
- epochs = 10: Sufficient convergence without overfitting

In [23]:
print(f"Using {len(training_sentences)} sentences for training")
print(f"This is {len(training_sentences)/len(sentences)*100:.1f}% of your total data")

model = Word2Vec(
    sentences=training_sentences,
    vector_size=200,
    window=5,
    min_count=10,
    workers=4,
    epochs=10,
    sg=1
)

model.save("dutch_custom_embeddings_final.model")

print(f"Training completed with {len(model.wv.index_to_key)} vocabulary words")

Using 3000000 sentences for training
This is 2.1% of your total data
Training completed with 63181 vocabulary words


### Observed Results:
- Successfully learned semantic clusters (e.g., "mooi" with "prachtig", "fraai")
- Identified challenge: antonyms co-occur in dialogue
